# Usage metering tests

In [ ]:
import boto3
import urllib.parse as urlparse 
import json
import shutil
import time

from datetime import datetime, timezone

In [ ]:
PROFILE = 'default'
REGION = 'us-east-1'

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

## Functions

Some functions to get product definition, put metering records in DynamoDB and scan a DynamoDB table.

In [ ]:

def get_marketplace_product(product_id):
    client = SESSION.client('marketplace-catalog')

    response = client.describe_entity(
    Catalog='AWSMarketplace',
    EntityId=product_id
    )
    
    return response

def create_metering_item(customer_aws_account_id, dimension_name, dimension_value):
    item = {
        "create_timestamp": {
            "N": f"{int(time.time())}"
        },
        "customerIdentifier": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": f'{dimension_value}'
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


def scan_table(table_name):
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    response = table.scan()
    return response['Items']


## Put metering records into DynamoDB

Put metering records (dimensions) into the metering table. You can get the metering table name from your CFN stack.

### Get DynamoDB table names
Get DynamoDB table names from the SaaS integration CFN stack.

Replace the value for `stack_name` with the name of you CFN stack.

Use `Metering table name` to ingest metering records.

In [ ]:
stack_name = 'eb-saas-con-sub'
cf_client = SESSION.client('cloudformation')
paginator = cf_client.get_paginator('list_stack_resources')


metering_table_name = None
lambda_hourly_function_name = None

for page in paginator.paginate(StackName=stack_name):
    for resource in page['StackResourceSummaries']:
        #print(json.dumps(resource, indent=2, default=str))
        #print(f"{resource['ResourceType']}: {resource['LogicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceSubscribers':
            print(f"Subscribers table name: {resource['PhysicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceMeteringRecords':
            print(f"Metering table name: {resource['PhysicalResourceId']}")
            metering_table_name = resource['PhysicalResourceId']
        if resource['LogicalResourceId'] == 'Hourly':
            print(f"Hourly Lambda function name: {resource['PhysicalResourceId']}")
            lambda_hourly_function_name = resource['PhysicalResourceId']


## Get product

Get your product description. In the description you find the usage dimension that you can use for metering.

In [ ]:
# replace the value for product_id with your product id
product_id = 'prod-xx3wopvpeyxqo'

print(json.dumps(get_marketplace_product(product_id), indent=2, default=str))

In [ ]:
# use the AWS account id for as your customer_identifier
customer_identifier = '944681004585'

In [ ]:
print(f"metering_table_name: {metering_table_name}")
print(f"lambda_hourly_function_name: {lambda_hourly_function_name}")
print(f"customer_identifier: {customer_identifier}")
ddb = SESSION.client('dynamodb')

## Create metering entries

Use `create_metering_item(CustomerAWSAccounId, Dimension)` to create
item and put them into the DynamoDB metering table.

In [ ]:
# put your usage identifiers into the list which you want to meter
usage_dimensions = [
    {'usage_1': 1},
    {'usage_2': 2}
]

for dimension in usage_dimensions:
    for dimension_name, dimension_value in dimension.items():
        print(f"metering dimension: {dimension_name} value: {dimension_value}")
        item = create_metering_item(customer_identifier, dimension_name, dimension_value)
        print(f"metering item:\n{json.dumps(item, indent=2, default=str)}")

        response = ddb.put_item(
            TableName=metering_table_name,
            Item=item
        )
        print(json.dumps(response, indent=2, default=str))
        print('-' * 50)
        time.sleep(2)

## Get entries from the metering table

Scan the metering table.

Unprocessed item look similar to:

```
{
  "dimension_usage": [
    {
      "dimension": "usage_2",
      "value": "3"
    }
  ],
  "metering_pending": "true",
  "create_timestamp": "1763396941",
  "customerIdentifier": "944681004585"
}
```

Processed records have a `metering_failed` boolean key and a `metering_response` key for example:

```
"metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "3"
    }
  ],
  "create_timestamp": "1763392067",
  "customerIdentifier": "944681004585",
  "metering_response": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"828fd9e5-ee21-4b0f-9656-82a86b4e49c2\",\"attempts\":1,\"totalRetryDelay\":0},\"Results\":[{\"MeteringRecordId\":\"eb0e9db9-c90e-45fa-84c4-6239a68fb360\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_1\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"6756c73c-21fd-4821-a5d5-57a6d891f103\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_2\",\"Quantity\":6,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"c65d412f-aedf-4e89-afa3-a1d238277b63\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_3\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}}],\"UnprocessedRecords\":[]}"
}
```

In [ ]:
# Usage
items = scan_table(metering_table_name)
for item in items:
    print(json.dumps(item, indent=2, default=str))
    print('-' * 50)

## Trigger metering hourly function

The Lambda function that meters hourly is automatically triggered by an
Amazon EventBridge rule. For testing purposes you can also
invoke the funtion manually.

In [ ]:
lmbd = SESSION.client('lambda')

response = lmbd.invoke(
    FunctionName=lambda_hourly_function_name,
    Payload=json.dumps({'start': 'metering'})
)

print(f"response:\n{json.dumps(response, indent=2, default=str)}")
print(f"Payload:\n{json.loads(response['Payload'].read())}")
